In [1]:
import os, glob, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATASET_ROOT = os.environ.get("DATASET_ROOT", "/home/jovyan/shared/donut/Alzheimer/OAS1")
SPLIT_SEED = 1334                         # ใช้ตอนสร้างไฟล์ครั้งเดียว ไม่ถูกเก็บในไฟล์
SPLIT_PATH = "split.json"
REFERENCE_PATH = "split_reference.json"   # snapshot เดิม (ถ้ามี จะใช้เทียบ)


In [2]:
df = pd.read_excel(f"{DATASET_ROOT}/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = f"{DATASET_ROOT}/{subject_id}/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    files = glob.glob(pattern)
    return files[0] if files else None

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
df["subject_id"] = df["ID"].apply(lambda x: x.split("_")[1])
print("df หลังกรอง:", df.shape, "| subject ไม่ซ้ำ:", df["subject_id"].nunique())


df หลังกรอง: (234, 15) | subject ไม่ซ้ำ: 234


In [3]:
sl = df.groupby("subject_id")["label"].first()
ids, vals = list(sl.index), list(sl.values)
tr, temp = train_test_split(ids, test_size=0.3, stratify=vals, random_state=SPLIT_SEED)
lab = dict(zip(ids, vals))
va, te = train_test_split(temp, test_size=0.5, stratify=[lab[s] for s in temp], random_state=SPLIT_SEED)
tr, va, te = sorted(tr), sorted(va), sorted(te)

split = {
    "labels": {s: int(lab[s]) for s in sorted(ids)},   # 0 = ปกติ, 1 = อัลไซเมอร์
    "train": tr, "val": va, "test": te,
}

if os.path.exists(SPLIT_PATH):
    print(f"มี {SPLIT_PATH} อยู่แล้ว -> ไม่เขียนทับ (ลบเองถ้าตั้งใจสร้างใหม่)")
else:
    with open(SPLIT_PATH, "w") as f:
        json.dump(split, f, indent=2, ensure_ascii=False)
    print(f"เขียน {SPLIT_PATH} แล้ว")

for k in ("train", "val", "test"):
    n_ad = sum(lab[s] == 1 for s in split[k])
    print(f"  {k:5s} คน={len(split[k])} ปกติ={len(split[k]) - n_ad} อัลไซเมอร์={n_ad}")


เขียน split.json แล้ว
  train คน=163 ปกติ=93 อัลไซเมอร์=70
  val   คน=35 ปกติ=20 อัลไซเมอร์=15
  test  คน=36 ปกติ=21 อัลไซเมอร์=15


In [4]:
if os.path.exists(REFERENCE_PATH):
    with open(REFERENCE_PATH) as f:
        ref = json.load(f)
    for k in ("train", "val", "test"):
        diff = set(ref[k]) ^ set(split[k])
        print(f"{k:5s}: {'ตรงกับ split_reference.json ทุกคน' if not diff else f'ต่างกัน {len(diff)} คน เช่น {sorted(diff)[:5]}'}")
else:
    print(f"ไม่พบ {REFERENCE_PATH} (ข้ามการเทียบ)")


train: ตรงกับ split_reference.json ทุกคน
val  : ตรงกับ split_reference.json ทุกคน
test : ตรงกับ split_reference.json ทุกคน


In [5]:
# ===== เซลล์โหลด: คัดลอกไปวางแทนเซลล์ split เดิมในโน้ตบุ๊กทุกโมเดล =====
# (ต้องมี df ที่มี subject_id และ label อยู่ก่อน)
import json

SPLIT_PATH = "split.json"      # <- ชี้ไปไฟล์กลางที่ทุกโมเดลใช้ร่วมกัน (เช่น โฟลเดอร์ shared)

def load_locked_split(df, path=SPLIT_PATH):
    # อ่านรายชื่อจาก split.json แล้วคืน train_df, val_df, test_df
    # df ต้องมีคอลัมน์ subject_id และ label (CDR>0) ถ้ารายชื่อหรือ label ไม่ตรงกับไฟล์จะ error ทันที
    with open(path) as f:
        sp = json.load(f)
    S = {k: set(sp[k]) for k in ("train", "val", "test")}

    # 1) ไม่มีชื่อซ้ำในชุดเดียว และไม่มี overlap ทีละคู่
    assert all(len(sp[k]) == len(S[k]) for k in S), "มีชื่อซ้ำในชุดเดียวกัน"
    assert not (S["train"] & S["val"]) and not (S["train"] & S["test"]) and not (S["val"] & S["test"]), "overlap ระหว่างชุด"
    # 2) รายชื่อใน df ต้องตรงกับในไฟล์ทุกคน (กันการกรองข้อมูลต่างกัน)
    in_df, in_file = set(df["subject_id"]), S["train"] | S["val"] | S["test"]
    assert in_df == in_file, (f"รายชื่อไม่ตรง: อยู่ใน df แต่ไม่อยู่ในไฟล์ {sorted(in_df - in_file)[:10]} "
                              f"| อยู่ในไฟล์แต่ไม่อยู่ใน df {sorted(in_file - in_df)[:10]}")
    # 3) label ใน df ต้องตรงกับในไฟล์ (กัน CDR/เกณฑ์ label เปลี่ยน)
    lab = df.groupby("subject_id")["label"].first()
    bad = [s for s in in_file if int(lab[s]) != int(sp["labels"][s])]
    assert not bad, f"label ไม่ตรงกับไฟล์ {len(bad)} คน เช่น {bad[:10]}"

    out = [df[df["subject_id"].isin(S[k])] for k in ("train", "val", "test")]
    print(f"โหลด {path}")
    for k, d in zip(("train", "val", "test"), out):
        n_ad = int((d["label"] == 1).sum())
        print(f"  {k:5s} คน={len(d)} ปกติ={len(d) - n_ad} อัลไซเมอร์={n_ad} ({100 * n_ad / len(d):.1f}%)")
    return tuple(out)

train_df, val_df, test_df = load_locked_split(df)


โหลด split.json
  train คน=163 ปกติ=93 อัลไซเมอร์=70 (42.9%)
  val   คน=35 ปกติ=20 อัลไซเมอร์=15 (42.9%)
  test  คน=36 ปกติ=21 อัลไซเมอร์=15 (41.7%)


In [6]:
import copy, tempfile, io, contextlib

def expect_fail(name, mutate_file=None, mutate_df=None):
    d = df.copy() if mutate_df is None else mutate_df(df.copy())
    sp = copy.deepcopy(json.load(open(SPLIT_PATH)))
    if mutate_file:
        mutate_file(sp)
    tmp = tempfile.NamedTemporaryFile("w", suffix=".json", delete=False)
    json.dump(sp, tmp); tmp.close()
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            load_locked_split(d, tmp.name)
        print(f"  {name}: *** ไม่ถูกจับ ***")
    except AssertionError as e:
        print(f"  {name}: ถูกจับได้ -> {str(e)[:80]}")

expect_fail("ใส่คนจาก train ซ้ำใน test (overlap)", lambda sp: sp["test"].append(sp["train"][0]))
expect_fail("ลบคนออกจากไฟล์ 1 คน", lambda sp: sp["train"].pop())
expect_fail("df ขาดคนไป 1 คน", mutate_df=lambda d: d[d["subject_id"] != d["subject_id"].iloc[0]])
expect_fail("label ใน df เปลี่ยน", mutate_df=lambda d: d.assign(label=np.where(d["subject_id"] == d["subject_id"].iloc[0], 1 - d["label"], d["label"])))

print()
print("โหลดไฟล์จริง (ต้องผ่าน):")
train_df, val_df, test_df = load_locked_split(df)


  ใส่คนจาก train ซ้ำใน test (overlap): ถูกจับได้ -> overlap ระหว่างชุด
  ลบคนออกจากไฟล์ 1 คน: ถูกจับได้ -> รายชื่อไม่ตรง: อยู่ใน df แต่ไม่อยู่ในไฟล์ ['0457'] | อยู่ในไฟล์แต่ไม่อยู่ใน df [
  df ขาดคนไป 1 คน: ถูกจับได้ -> รายชื่อไม่ตรง: อยู่ใน df แต่ไม่อยู่ในไฟล์ [] | อยู่ในไฟล์แต่ไม่อยู่ใน df ['0001'
  label ใน df เปลี่ยน: ถูกจับได้ -> label ไม่ตรงกับไฟล์ 1 คน เช่น ['0001']

โหลดไฟล์จริง (ต้องผ่าน):
โหลด split.json
  train คน=163 ปกติ=93 อัลไซเมอร์=70 (42.9%)
  val   คน=35 ปกติ=20 อัลไซเมอร์=15 (42.9%)
  test  คน=36 ปกติ=21 อัลไซเมอร์=15 (41.7%)
